In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Load the dataset
df = pd.read_csv("ETT-small/ETTh1.csv")

# Convert date column into datetime format
df["date"] = pd.to_datetime(df["date"])

# Display dataset information
print("Dataset shape:", df.shape)

print("\nFirst five rows:")
display(df.head())

print("\nLast five rows:")
display(df.tail())

Dataset shape: (17420, 8)

First five rows:


,date,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT
0,2016-07-01 00:00:00,5.827,2.009,1.599,0.462,4.203,1.340,30.531000
1,2016-07-01 01:00:00,5.693,2.076,1.492,0.426,4.142,1.371,27.787001
2,2016-07-01 02:00:00,5.157,1.741,1.279,0.355,3.777,1.218,27.787001
3,2016-07-01 03:00:00,5.090,1.942,1.279,0.391,3.807,1.279,25.044001
4,2016-07-01 04:00:00,5.358,1.942,1.492,0.462,3.868,1.279,21.948000



Last five rows:


,date,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT
17415,2018-06-26 15:00:00,-1.674,3.550,-5.615,2.132,3.472,1.523,10.904
17416,2018-06-26 16:00:00,-5.492,4.287,-9.132,2.274,3.533,1.675,11.044
17417,2018-06-26 17:00:00,2.813,3.818,-0.817,2.097,3.716,1.523,10.271
17418,2018-06-26 18:00:00,9.243,3.818,5.472,2.097,3.655,1.432,9.778
17419,2018-06-26 19:00:00,10.114,3.550,6.183,1.564,3.716,1.462,9.567


## 2. Chronological Data Splitting

The ETTh1 dataset is divided into training, validation, and testing
sets using the standard ETTh1 benchmark convention.

- Training: First 12 months (8,640 observations)
- Validation: Next 4 months (2,880 observations)
- Testing: Final 4 months (2,880 observations)

Chronological splitting preserves the temporal order of observations
and prevents future observations from being used during model training.

The standard benchmark uses the first 14,400 observations.

In [3]:
# Define the standard ETTh1 split boundaries
train_end = 12 * 30 * 24
val_end = train_end + 4 * 30 * 24
test_end = val_end + 4 * 30 * 24

# Split the dataset chronologically
train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:test_end].copy()

# Display split information
for name, data in [
    ("Training", train_df),
    ("Validation", val_df),
    ("Testing", test_df)
]:
    print(f"\n{name} dataset")
    print("Shape:", data.shape)
    print("Start date:", data["date"].iloc[0])
    print("End date:", data["date"].iloc[-1])


Training dataset
Shape: (8640, 8)
Start date: 2016-07-01 00:00:00
End date: 2017-06-25 23:00:00

Validation dataset
Shape: (2880, 8)
Start date: 2017-06-26 00:00:00
End date: 2017-10-23 23:00:00

Testing dataset
Shape: (2880, 8)
Start date: 2017-10-24 00:00:00
End date: 2018-02-20 23:00:00


## 3. Feature Normalization

StandardScaler is used to standardize the seven numerical
features of ETTh1.

The scaler is fitted only on the training dataset to avoid
data leakage. The same scaler is then applied to the
training, validation, and testing datasets.

In [4]:
from sklearn.preprocessing import StandardScaler

# Select the seven numerical features
feature_cols = [
    "HUFL", "HULL", "MUFL",
    "MULL", "LUFL", "LULL", "OT"
]

# Create the scaler
scaler = StandardScaler()

# Learn mean and standard deviation from training data only
scaler.fit(train_df[feature_cols])

# Apply the same scaler to all three datasets
train_scaled = scaler.transform(train_df[feature_cols])
val_scaled = scaler.transform(val_df[feature_cols])
test_scaled = scaler.transform(test_df[feature_cols])

# Check results
print("Training shape:", train_scaled.shape)
print("Validation shape:", val_scaled.shape)
print("Testing shape:", test_scaled.shape)

print("\nTraining feature means:")
print(np.round(train_scaled.mean(axis=0), 4))

print("\nTraining feature standard deviations:")
print(np.round(train_scaled.std(axis=0), 4))

Training shape: (8640, 7)
Validation shape: (2880, 7)
Testing shape: (2880, 7)

Training feature means:
[ 0. -0.  0. -0.  0. -0.  0.]

Training feature standard deviations:
[1. 1. 1. 1. 1. 1. 1.]


## 4. Inverse Normalization

Inverse normalization converts standardized values back
to their original measurement scale.

This is important because forecasting models will produce
predictions in standardized units, while we may want to
evaluate and visualize oil temperature predictions in
their original units.

The original values can be recovered using the training
mean and standard deviation.

In [5]:
# Find the column position of oil temperature
ot_index = feature_cols.index("OT")

# Get OT mean and standard deviation from training data
ot_mean = scaler.mean_[ot_index]
ot_std = scaler.scale_[ot_index]

# Select the first five normalized OT values
normalized_ot = train_scaled[:5, ot_index]

# Convert them back to original values
recovered_ot = normalized_ot * ot_std + ot_mean

# Compare with original values
comparison = pd.DataFrame({
    "Original OT": train_df["OT"].iloc[:5].values,
    "Normalized OT": normalized_ot,
    "Recovered OT": recovered_ot
})

print(comparison.round(4))

   Original OT  Normalized OT  Recovered OT
0       30.531         1.4606        30.531
1       27.787         1.1615        27.787
2       27.787         1.1615        27.787
3       25.044         0.8626        25.044
4       21.948         0.5252        21.948


## 5. Sliding Window Sequence Generation

Sliding windows convert time-series observations into
supervised learning examples.

For the initial forecasting experiment:

- Input sequence length: 96 hours
- Prediction horizon: 96 hours
- Input features: All seven numerical variables
- Prediction target: Oil Temperature (OT)
- Sliding window stride: 1 hour

Each input contains historical observations, while
the corresponding output contains future oil temperature
values.

This approach will be extended to prediction horizons
of 192, 336, and 720 hours.

In [6]:
# Define input and prediction lengths
input_len = 96
pred_len = 96

# Select the first 96 hours as input
X_example = train_scaled[0:input_len, :]

# Select the following 96 hours of OT as output
y_example = train_scaled[
    input_len:input_len + pred_len,
    ot_index
]

# Display shapes
print("Input shape:", X_example.shape)
print("Output shape:", y_example.shape)

# Display the first five OT values
print("\nFirst five input OT values:")
print(X_example[:5, ot_index])

print("\nFirst five target OT values:")
print(y_example[:5])

Input shape: (96, 7)
Output shape: (96,)

First five input OT values:
[1.46055158 1.16152666 1.16152666 0.86261065 0.52522672]

First five target OT values:
[0.96221295 0.96221295 0.9698411  1.28423145 1.4528143 ]


## 6. Sliding Window Generation

A reusable Python function is used to generate multiple
input-output sequences from the normalized time-series data.

Each input sequence contains 96 historical hourly observations
of all seven features.

Each output sequence contains the following 96 hourly
oil temperature (OT) values.

The sliding window moves forward by one hour at a time.

In [7]:
def create_sequences(data, input_len, pred_len, target_index):

    X = []
    y = []

    for i in range(len(data) - input_len - pred_len + 1):

        # Select historical input observations
        input_sequence = data[i:i + input_len, :]

        # Select future oil temperature values
        output_sequence = data[
            i + input_len:i + input_len + pred_len,
            target_index
        ]

        X.append(input_sequence)
        y.append(output_sequence)

    return np.array(X), np.array(y)

In [8]:
# Generate training sequences
X_train, y_train = create_sequences(
    train_scaled,
    input_len=96,
    pred_len=96,
    target_index=ot_index
)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("\nTotal training examples:", len(X_train))

X_train shape: (8449, 96, 7)
y_train shape: (8449, 96)

Total training examples: 8449


## 7. Validation and Test Sequence Generation

Validation and testing sequences are generated using
the same sliding-window approach as training.

The input window is allowed to include historical
observations from the preceding dataset split.

However, all prediction targets must remain within
their respective validation or testing periods.

This preserves chronological order and prevents
future target leakage.

In [9]:
# Include the previous 96 hours as historical context
val_with_history = np.concatenate(
    [train_scaled[-input_len:], val_scaled],
    axis=0
)

test_with_history = np.concatenate(
    [val_scaled[-input_len:], test_scaled],
    axis=0
)

# Generate validation sequences
X_val, y_val = create_sequences(
    val_with_history,
    input_len,
    pred_len,
    ot_index
)

# Generate testing sequences
X_test, y_test = create_sequences(
    test_with_history,
    input_len,
    pred_len,
    ot_index
)

# Display results
print("Validation input shape:", X_val.shape)
print("Validation output shape:", y_val.shape)

print("\nTesting input shape:", X_test.shape)
print("Testing output shape:", y_test.shape)

Validation input shape: (2785, 96, 7)
Validation output shape: (2785, 96)

Testing input shape: (2785, 96, 7)
Testing output shape: (2785, 96)


## 8. Verification of Sliding Window Boundaries

The generated sequences are checked to ensure that
historical input observations and future prediction
targets are correctly aligned.

The first validation input should match the last 96
training observations, while its target should match
the first 96 validation oil temperature values.

The same verification is performed for the testing set.

These checks help prevent incorrect sequence alignment
and future information leakage.

In [10]:
# Verify the first validation window
val_input_correct = np.allclose(
    X_val[0],
    train_scaled[-input_len:]
)

val_target_correct = np.allclose(
    y_val[0],
    val_scaled[:pred_len, ot_index]
)

# Verify the first testing window
test_input_correct = np.allclose(
    X_test[0],
    val_scaled[-input_len:]
)

test_target_correct = np.allclose(
    y_test[0],
    test_scaled[:pred_len, ot_index]
)

# Display verification results
print("Validation input correct:", val_input_correct)
print("Validation target correct:", val_target_correct)

print("\nTesting input correct:", test_input_correct)
print("Testing target correct:", test_target_correct)

Validation input correct: True
Validation target correct: True

Testing input correct: True
Testing target correct: True


## 9. Preparing Data for PyTorch

The preprocessed NumPy arrays are converted into
PyTorch tensors for neural network training.

The tensors use float32 precision, which is commonly
used for deep learning and GPU computation.

The training, validation, and testing datasets are
converted separately to preserve the existing data splits.

In [12]:
import torch

# Convert training arrays into tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)

# Convert validation arrays into tensors
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32)

# Convert testing arrays into tensors
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

# Display results
print("PyTorch version:", torch.__version__)

print("\nTraining tensors:")
print("X:", X_train_tensor.shape)
print("y:", y_train_tensor.shape)

print("\nValidation tensors:")
print("X:", X_val_tensor.shape)
print("y:", y_val_tensor.shape)

print("\nTesting tensors:")
print("X:", X_test_tensor.shape)
print("y:", y_test_tensor.shape)

print("\nData type:", X_train_tensor.dtype)

PyTorch version: 2.3.1+cu118

Training tensors:
X: torch.Size([8449, 96, 7])
y: torch.Size([8449, 96])

Validation tensors:
X: torch.Size([2785, 96, 7])
y: torch.Size([2785, 96])

Testing tensors:
X: torch.Size([2785, 96, 7])
y: torch.Size([2785, 96])

Data type: torch.float32


In [11]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.3.1+cu118
CUDA available: True


## 10. Creating PyTorch Datasets and DataLoaders

The input and target tensors are combined into PyTorch datasets.

Each dataset contains pairs of input sequences and corresponding forecasting targets. DataLoaders will later be used to organize these samples into batches for model training and evaluation.

In [13]:
from torch.utils.data import TensorDataset

# Create datasets from input and target tensors
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# Check the number of samples
print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Testing samples:", len(test_dataset))

Training samples: 8449
Validation samples: 2785
Testing samples: 2785


In [14]:
from torch.utils.data import DataLoader

# Define batch size
batch_size = 32

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Check the number of batches
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

Training batches: 265
Validation batches: 88
Testing batches: 88


In [15]:
# Get one batch from the training DataLoader
X_batch, y_batch = next(iter(train_loader))

# Display batch information
print("Input batch shape:", X_batch.shape)
print("Target batch shape:", y_batch.shape)
print("Input data type:", X_batch.dtype)
print("Target data type:", y_batch.dtype)

Input batch shape: torch.Size([32, 96, 7])
Target batch shape: torch.Size([32, 96])
Input data type: torch.float32
Target data type: torch.float32
